# RAG Architecture Mechanisms

*What each named pattern buys, and where it fails.*

Every survey of RAG architectures is a table: the names down the left, and one column across the
top — **best for**. There is never a second column, and the second column is the one that decides
anything.

This notebook supplies it. Six architectures run as arms over two regimes, and we measure not only
the class each one wins but the class each one loses, attributing both to a named property of the
corpus. It imports `rag_architecture_mechanisms.py`, which owns every number; nothing here is
recomputed.

Run it with:

```
uv run --with numpy --with scipy --with jupyter \
    jupyter execute notebooks/rag-architecture-mechanisms/01_rag_architecture_mechanisms.ipynb
```


In [ ]:
import pathlib
import sys

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / "rag_architecture_mechanisms.py").exists()
                       else pathlib.Path.cwd() / "notebooks" / "rag-architecture-mechanisms"))
import rag_architecture_mechanisms as M

L, G = M.local(), M.glob()
print(f"LOCAL : {L['n_queries']} queries over {L['n_docs']} passages of {L['K']} companies, dim {M.DIM}")
print(f"GLOBAL: {G['n_queries']} themes over the same {G['K']} entities in {M.N_SECTORS} sectors")
print(f"arms  : {', '.join(M.ARMS)}")

## 1. Two places an answer can live

An architecture can only be right *about a corpus*, so we build two.

In the **local** regime the answer is carried by a passage — some filing says the thing, and the
job is to find it. In the **global** regime the answer is carried by no passage at all: the
question is which sector discusses a theme most, and that is a count across a community.

The global regime is easy to build badly. If the sector that discusses a theme most were also the
sector nearest it, ranking entities by similarity would already answer the question and community
summarization would be decoration. So we decouple the count from the geometry — and assert it.

In [ ]:
disagree = int((G["gold"] != G["geo_answer"]).sum())
print(f"themes where the count-max sector is NOT the nearest sector: {disagree}/{G['n_queries']}")

labels = M.communities(G)
from collections import Counter
pure = all(len(Counter(G["sector_of"][labels == c])) == 1 for c in range(labels.max() + 1))
print(f"Leiden recovers {labels.max() + 1} communities; each pure by sector: {pure}")

flat = M.run_arm(G, "naive")["acc"]
print(f"flat top-{M.GLOBAL_TOPK} sector voting: {flat:.3f}   (guessing among {M.N_SECTORS} would be {1 / M.N_SECTORS:.2f})")

That last number is below chance, and deliberately so. A geometric arm on this regime is not
merely uninformed, it is systematically drawn to the wrong sector.

## 2. Why the local query classes separate

The local corpus needs five query classes, one per mechanism. Our first attempt used the query's
*concentration* as the difficulty knob — sharply named queries easy, vaguely named ones hard — and
the classes refused to separate.

The reason is that what determines whether a single view can identify a company is not how sharply
the query names a theme but **how many other companies share that theme**. Two classes differing
only in concentration are one mechanism sampled twice.

So the sharing structure is constructed rather than drawn. Two company families:

In [ ]:
ct, dist = M.theme_assignment()
share = np.array([[int((ct[:, f] == ct[a, f]).sum()) for f in range(3)] for a in range(len(ct))])
for name, mask in (("distinctive", dist == 1), ("conjunctive", dist == 0)):
    print(f"{name:12s} n={int(mask.sum()):2d}  companies sharing its theme, per view: "
          f"{share[mask].mean(axis=0).round(2)}")
print()
print("A distinctive company holds a DENSE theme no one else holds — one view identifies it.")
print("A conjunctive company holds only shared themes — only the conjunction of three does.")

### The bug that structure was hiding

The three legs are supposed to read **disjoint** token windows, so that they are partial views of
one document rather than three noisy copies of one score. In the first build the late window
overlapped the other two — and because a passage is filled window by window, the late draw silently
overwrote half the dense window.

The effect was not noise but a phantom result: the dense leg scored a perfect 1.000 on the very
class built to defeat it. Nothing checked the docstring's claim, so now something does.

In [ ]:
M.test_window_partition()
print("windows:", M.WINDOWS, "-> partition of", M.TOKENS, "tokens: OK")
print("theme triples unique across all companies: OK")

## 3. Fusion is set intersection

The hybrid arm fuses three legs by reciprocal rank. What can that buy? Only disagreement — if the
legs agree, the fused order is the shared order.

So fusion should be decisive exactly where no single view identifies the answer, and a *liability*
where one already does. Both halves are measurable.

In [ ]:
for k in ("partial", "on"):
    m = M._class_idx(L, k)
    legs = {n: sum(M.answer_of(L, np.argsort(-fn(L, L["Q"][i]))) == L["gold"][i] for i in m) / len(m)
            for n, fn in M.LEGS.items()}
    fused = sum(M.answer_of(L, M.rrf_fuse(M._legs_rankings(L, L["Q"][i]))) == L["gold"][i]
                for i in m) / len(m)
    verdict = "fusion WINS" if fused > max(legs.values()) else "fusion LOSES to its own best leg"
    print(f"{k:8s} legs " + "  ".join(f"{n}={v:.2f}" for n, v in legs.items())
          + f"   fused={fused:.2f}   <- {verdict}")

On the conjunctive class each leg returns rivals and their intersection is the answer: +0.65 over
the best leg. On the class one view answers outright, two uninformed legs outvote the informed one
and fusion is strictly worse than not fusing. Hybrid is a trade, not an upgrade.

Reaching the winning half took three constructions, and the two failures stay in the module as
controls: legs carrying different *noise* on the same content never robustly win. Fusion needs the
views to carry different **content**.

## 4. Rank against reach

The corrective arm retrieves cheaply, grades the result, and on a bad grade over-fetches and
re-scores with MaxSim over every token. A cascade can only reorder what it fetched, which gives
correction a sharp boundary: it moves rank, it cannot create reach.

In [ ]:
for k in M.CLASSES:
    ranks = np.array([M.gold_rank(L, M.leg_dense(L, L["Q"][i]), L["gold"][i]) for i in M._class_idx(L, k)])
    print(f"{k:8s} gold's median rank in the cheap ranking: {np.median(ranks):5.1f}"
          f"   within the over-fetch depth of {M.OVER_FETCH}: {float((ranks < M.OVER_FETCH).mean()):.2f}")

mm = M.mechanism_matrix()
print()
print(f"corrective on 'noisy'  (present but demoted): {mm['corrective']['local']['per_class']['noisy']:.3f}"
      f"   vs naive {mm['naive']['local']['per_class']['noisy']:.3f}")
print(f"corrective on 'bridge' (no passage is about it): {mm['corrective']['local']['per_class']['bridge']:.3f}")

### The grader cannot warn you

The grader is a relevance judge, and a good one — AUC 0.829 against *was the cheap answer right*.
On the bridge class it is blind, and it is blind for a reason no calibration fixes: there the
retrieved filing genuinely **is** highly relevant. It is the filing of the company the query
describes. The answer is simply a different company, one that filing merely names.

In [ ]:
for k in ("on", "bridge"):
    g = float(np.mean([M.grade(L, i, M.arch_naive(L, i)["ranking"]) for i in M._class_idx(L, k)]))
    print(f"{k:8s} mean grade {g:.3f}   naive accuracy {mm['naive']['local']['per_class'][k]:.2f}")
print()
print("Same grade. One class is answered perfectly, the other not at all.")
print("Relevance is not correctness when the answer is a company no passage is about.")

## 5. Generation corrects position, not content

HyDE throws the query's position away and retrieves with a generated document instead. Our
generator never sees the gold: it conditions on the query and drops the component along the
corpus-mean direction $g$ — the axis separating queries from documents, which by construction says
nothing about *which* company.

That single move is both the whole correction and the whole limitation.

In [ ]:
for k in ("off", "noisy"):
    print(f"{k:8s} naive {mm['naive']['local']['per_class'][k]:.2f}  ->  hyde {mm['hyde']['local']['per_class'][k]:.2f}")
print()
print("'off'   is tilted along g          -> removing g restores it")
print("'noisy' is tilted toward a rival   -> removing g does nothing at all")
print()
M.test_hyde_alpha_zero_is_naive()
print("collapse anchor: hyde(alpha=0) reproduces naive's ranking exactly")

## 6. Only a second hop reaches what is named

A bridge passage is a filing of $x$ that names $y$: its window is
$\cos\alpha\,\theta(x) + \sin\alpha\,\theta(y)$, so the named company survives only in the
component **orthogonal** to the filing it sits in. Ranking by similarity reads the $\cos\alpha$
part and discards the rest.

The agentic arm applies the imported reformulation operator to recover it — and pays for that bet
on every query, including the ones where it was wrong to make it.

In [ ]:
print(f"agentic on 'bridge': {mm['agentic']['local']['per_class']['bridge']:.3f}")
for a in ("naive", "hybrid", "hyde", "corrective", "graph"):
    print(f"  {a:11s} {mm[a]['local']['per_class']['bridge']:.3f}")
print()
print(f"full MaxSim over EVERY token of every document also reaches it: "
      f"{sum(M.answer_of(L, np.argsort(-M.full_maxsim(L, L['Q'][i]))) == L['gold'][i] for i in M._class_idx(L, 'bridge'))}/20")
print()
print(f"and the price: agentic hops {mm['agentic']['local']['mean_hops']:.2f} times on average, and is the")
print(f"WORST arm overall locally at {mm['agentic']['local']['acc']:.3f} against naive's {mm['naive']['local']['acc']:.3f}.")
print(f"On the class naive answers perfectly it scores {mm['agentic']['local']['per_class']['on']:.2f}.")

The stopping rule asks whether the filing just read opens a direction the query did not have. It
cannot tell a filing that *names a new company* from one that is merely off-target — both look like
a large residual. Nothing in the retrieval signal says a second hop is needed; on a bridge query the
top hit is the source company's own filing, confidently retrieved, and the query looks answered.

## 7. Aggregation is precomputed or paid for

GraphRAG is the only arm holding a representation of a **set**. It takes the global regime outright.

The honest form of that result is a cost, not an impossibility: flat retrieval *can* reach the same
answer by scanning far enough down and keeping the entities that discuss the theme — which is the
aggregation a community summary holds, done by hand at query time.

In [ ]:
print("flat aggregation by hand, in the global regime:")
for d, a in M.flat_aggregation_curve():
    bar = "#" * int(round(a * 40))
    print(f"  depth {d:2d}  {a:.3f}  {bar}")
print()
print(f"  graph      {M.run_arm(G, 'graph')['acc']:.3f}  at depth 0, because it paid once, offline")
print()
print(f"It reaches the answer at depth {G['K']}, which is the ENTIRE entity set.")

## 8. The matrix

Six arms, two regimes, both halves of each.

In [ ]:
print(M._fmt_matrix(mm))

Read the columns, not the third decimals: twenty queries per class means one query is 0.05, so
the 0.10 between corrective and hybrid on the noisy class is two queries and we decline to call it
a ranking.

There is **no dominant row**. The arm that takes the global regime is mid-table locally. The arm
that owns the bridge class is the worst arm locally overall. The baseline is unimprovable on one
class and last on three. Every arm is best at exactly one thing, and that thing is a property of
the corpus we can name in a sentence.

## 9. What the average hides

We went in expecting to show that over-eager correction is a net loss. It is not — and reporting
that is more useful than fixing it.

In [ ]:
grid = M.correction_damage_grid()
naive_acc = M.run_arm(L, "naive")["acc"]
print(f"naive baseline: {naive_acc:.3f}")
print(f"{'grader mid':>11s} {'sigma':>6s} {'overall':>8s} {'on-class':>9s} {'fires':>6s}")
for (mid, sg), (acc, on, fire) in sorted(grid.items()):
    flag = "  <- perfect class destroyed" if on < 1.0 else ""
    print(f"{mid:11.2f} {sg:6.1f} {acc:8.3f} {on:9.2f} {fire:6.2f}{flag}")
print()
print("Correcting is never a NET loss here: even at the worst corner it beats the baseline,")
print("because that baseline is weak enough that a badly reordered shortlist is an average gain.")
print("What it does instead is destroy the one class that needed no correcting, while the mean rises.")

This is the aggregation trap in its purest form, and it is the same trap the *best for* column
sets. An architecture evaluated on a mixture reports what the mixture rewards. Change the mixture
and a change that measured as an improvement becomes a regression, with nothing in the evaluation
having moved.

## 10. Every claim, as an assertion

Each pedagogical claim above is a test, including the ones that record a prediction failing.

In [ ]:
M._run_tests()